In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl

In [2]:
covid = pd.read_csv("./covid_toy.csv")
covid.sample(10)

,age,gender,fever,cough,city,has_covid
9,64,Female,101.0,Mild,Delhi,No
99,10,Female,98.0,Strong,Kolkata,Yes
85,16,Female,103.0,Mild,Bangalore,Yes
23,80,Female,98.0,Mild,Delhi,Yes
13,64,Male,102.0,Mild,Bangalore,Yes
92,82,Female,102.0,Strong,Kolkata,No
63,10,Male,100.0,Mild,Bangalore,No
76,80,Male,100.0,Mild,Bangalore,Yes
3,31,Female,98.0,Mild,Kolkata,No
58,23,Male,98.0,Strong,Mumbai,Yes


In [3]:
covid["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [4]:
covid["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

gender and city are Nominal Catagorical Variables and cough is Ordinal Catagorical Variable

In [5]:
covid.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
x_train, x_test, y_train, y_test = skl.model_selection.train_test_split(covid.drop(columns=["has_covid"]), covid["has_covid"], test_size= 0.2)

# Aam Zindagi

In [20]:
# We used the SimpleImputer on Fever as fever has null rows

si = skl.impute.SimpleImputer()

x_train_fever = si.fit_transform(x_train[["fever"]])
x_test_fever = si.fit_transform(x_test[["fever"]])

x_train_fever.shape

(80, 1)

In [21]:
oe = skl.preprocessing.OrdinalEncoder(categories= [["Mild", "Strong"]])
x_train_cough = oe.fit_transform(x_train[["cough"]])
x_test_cough = oe.fit_transform(x_test[["cough"]])

x_train_cough.shape

(80, 1)

In [22]:
ohe = skl.preprocessing.OneHotEncoder(drop= "first", sparse_output= False)

x_train_gender_city = ohe.fit_transform(x_train[["gender", "city"]])
x_test_gender_city = ohe.fit_transform(x_test[["gender", "city"]])

In [23]:
x_train_age = x_train.drop(columns=["gender", "fever", "cough", "city"])
x_test_age = x_test.drop(columns=["gender", "fever", "cough", "city"])

x_train_age.shape

(80, 1)

In [24]:
x_train_transformed = np.concatenate((x_train_age, x_train_fever, x_train_gender_city, x_train_cough), axis= 1)
x_test_transformed = np.concatenate((x_test_age, x_test_fever, x_test_gender_city, x_test_cough), axis= 1)

x_train_transformed.shape

(80, 7)

# Mentos Zindagi

In [31]:
transformer = skl.compose.ColumnTransformer(transformers=[
    ("tnf1", skl.impute.SimpleImputer(), ["fever"]),
    ("tnf2", skl.preprocessing.OrdinalEncoder(categories= [["Mild", "Strong"]]), ["cough"]),
    ("tnf3", skl.preprocessing.OneHotEncoder(drop= "first", sparse_output= False, dtype= np.int32), ["gender", "city"])
    ], remainder= "passthrough")  # remainder= drop

x_train_transformer = transformer.fit_transform(x_train)
x_test_transformer = transformer.fit_transform(x_test)
x_test_transformer.shape

(20, 7)